# bjy819 — Monthly activity and gaps

Run from the repository folder with pandas and matplotlib installed. This notebook uses the saved Part 1 CSV and retains reviewed text fields in the final stats file.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
Path("commits_timeseries").mkdir(exist_ok=True)
Path("graphs").mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(
    "bjy819_project_summary.csv",
    sep=";",
    keep_default_na=False
)

df = df.rename(columns={
    "project_wocid": "project",
    "commit_sha1": "commit",
    "commit message": "message"
})

df["Month"] = pd.to_datetime(
    df["time"], unit="s", utc=True
).dt.strftime("%Y-%m")

df.head()

In [3]:
for prj in df["project"].unique():
    # Monthly counts — same approach as the starter
    df1 = df[df["project"] == prj]
    df2 = df1.groupby("Month").size().reset_index(name="Commits")
    monthly_counts = df2.copy()

    monthly_counts["time"] = pd.to_datetime(monthly_counts["Month"])

    full_date_range = pd.DataFrame({
        "time": pd.date_range(
            start=monthly_counts["time"].min(),
            end=monthly_counts["time"].max(),
            freq="MS"
        )
    })

    monthly_counts = pd.merge(
        full_date_range, monthly_counts, on="time", how="left"
    )

    monthly_counts["Commits"] = (
        monthly_counts["Commits"].fillna(0).astype(int)
    )
    monthly_counts["Month"] = (
        monthly_counts["time"].dt.strftime("%Y-%m")
    )

    # Save the required two-column monthly CSV
    monthly_counts[["Month", "Commits"]].rename(
        columns={"Commits": "#Commits"}
    ).to_csv(
        "commits_timeseries/bjy819_commits_timeseries_" + prj + ".csv",
        sep=";",
        index=False
    )

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(
        monthly_counts["Month"],
        monthly_counts["Commits"],
        marker="o",
        linestyle="-"
    )
    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(prj)
    plt.grid(True)

    # Show approximately 12 date labels for readability
    step = max(1, len(monthly_counts) // 12)
    plt.xticks(
        range(0, len(monthly_counts), step),
        monthly_counts["Month"].iloc[::step],
        rotation=45,
        fontsize=7
    )

    plt.tight_layout()
    plt.savefig("graphs/bjy819_timeseries_" + prj + ".png", dpi=600)
    plt.close()

    print(f"Saved monthly CSV and graph for {prj}")

Saved monthly CSV and graph for mobvoi_wenet
Saved monthly CSV and graph for pbs-assess_gfsynopsis
Saved monthly CSV and graph for pmlmodelling_ersem
Saved monthly CSV and graph for vearch_vearch
Saved monthly CSV and graph for hannorein_rebound
Saved monthly CSV and graph for soedinglab_mmseqs2
Saved monthly CSV and graph for hpddm_hpddm
Saved monthly CSV and graph for stephenslab_mr.mash.alpha
Saved monthly CSV and graph for gem_oq-engine
Saved monthly CSV and graph for sixin-zh_kymatio_wph


## Longest inactivity gap

A gap is a consecutive run of zero-commit months between the first and last commit months. Ties use the earliest gap. NumberOfGapsInTimeline counts only runs of at least three months. NumCommitsAfterLastGap means commits after the selected longest gap.

In [4]:
github = pd.read_csv("bjy819_github_metadata.csv", sep=";")
project_stats = []

for prj in df["project"].unique():
    monthly = pd.read_csv(
        f"commits_timeseries/bjy819_commits_timeseries_{prj}.csv",
    sep=";"
    ).sort_values("Month").reset_index(drop=True)

    gaps = []
    gap_start = None

    for i, count in enumerate(monthly["#Commits"]):
        if count == 0:
            if gap_start is None:
                gap_start = i
        elif gap_start is not None:
            gaps.append((gap_start, i - 1))
            gap_start = None

    # Handle a gap reaching the last row, if present
    if gap_start is not None:
        gaps.append((gap_start, len(monthly) - 1))

    if gaps:
        # For ties, choose the earliest longest gap
        start, end = max(gaps, key=lambda gap: gap[1] - gap[0] + 1)

        longest_start = monthly.loc[start, "Month"]
        longest_end = monthly.loc[end, "Month"]
        longest_length = end - start + 1
        commits_after = int(monthly.loc[end + 1:, "#Commits"].sum())
    else:
        longest_start = ""
        longest_end = ""
        longest_length = 0
        commits_after = pd.NA  # No gap to count after

    number_of_gaps = sum(
        end - start + 1 >= 3
        for start, end in gaps
    )

    project = df[df["project"] == prj]
    dates = pd.to_datetime(project["time"], unit="s", utc=True)

    project_stats.append({
        "Project": prj,
        "ncommits": project["commit"].nunique(),
        "nauthors": project["author"].nunique(),
        "from": dates.min().isoformat(),
        "to": dates.max().isoformat(),
        "LongestGapStart": longest_start,
        "LongestGapEnd": longest_end,
        "LongestGapLength": longest_length,
        "NumCommitsAfterLastGap": commits_after,
        "NumberOfGapsInTimeline": number_of_gaps,
        "ActivityPattern": ""
    })

stats = pd.DataFrame(project_stats)

stats = stats.merge(
    github[["Project", "nstars", "nforks", "lastGHCommitDate"]],
    on="Project",
    how="left",
    validate="one_to_one"
)

stats["NumCommitsAfterLastGap"] = (
    stats["NumCommitsAfterLastGap"].astype("Int64")
)

# Refresh numerical results without erasing the reviewed text columns.
saved = pd.read_csv("bjy819_project_stats.csv", sep=";", keep_default_na=False)
text_columns = ["ActivityPattern", "BeforeThemes", "AfterThemes", "HypothesizedGapReason",
                "HasRecovered", "WhyRecovered", "WhoRecovered", "Currentstatus", "RecentThemes", "Notes"]
stats = stats.drop(columns=["ActivityPattern"]).merge(
    saved[["Project"] + text_columns], on="Project", validate="one_to_one")
stats = stats[saved.columns]
stats.to_csv("bjy819_project_stats.csv", sep=";", index=False)

print(stats[[
    "Project",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "NumberOfGapsInTimeline"
]])

                     Project  ... NumberOfGapsInTimeline
0               mobvoi_wenet  ...                      0
1      pbs-assess_gfsynopsis  ...                      3
2         pmlmodelling_ersem  ...                      9
3              vearch_vearch  ...                      2
4          hannorein_rebound  ...                      1
5         soedinglab_mmseqs2  ...                      0
6                hpddm_hpddm  ...                      2
7  stephenslab_mr.mash.alpha  ...                      2
8              gem_oq-engine  ...                      0
9       sixin-zh_kymatio_wph  ...                      6

[10 rows x 6 columns]


In [5]:
# Last 10 before and first 10 after the longest gap (all if fewer exist).
parts = []
for _, row in stats.iterrows():
    if int(row["LongestGapLength"]) == 0:
        continue  # gem_oq-engine has no gap; no artificial pre/post rows.
    project = df[df["project"] == row["Project"]].sort_values(["time", "commit"])
    for label, sample in [
        ("pre", project[project["Month"] < row["LongestGapStart"]].tail(10)),
        ("post", project[project["Month"] > row["LongestGapEnd"]].head(10))
    ]:
        sample = sample[["project", "commit", "author", "time", "message"]].copy()
        sample.insert(0, "pre/post", label)
        parts.append(sample)
gap_commits = pd.concat(parts, ignore_index=True).rename(columns={
    "project": "project_wocid", "commit": "commit_sha1", "message": "commit message"})
gap_commits.to_csv("bjy819_project_gap_commits.csv", sep=";", index=False)
print("Saved", len(gap_commits), "gap sample commits.")

Saved 172 gap sample commits.


## Interpretation

**mobvoi_wenet**: Activity falls after the 2021 peak, supporting declining. The longest gap lasted 1 months, with 0 gaps of at least three months. The gap cause is difficult to establish from the messages.

**pbs-assess_gfsynopsis**: Activity varies in bursts, including a substantial 2023 rebound, supporting irregular. The longest gap lasted 5 months, with 3 gaps of at least three months. The gap cause is difficult to establish from the messages.

**pmlmodelling_ersem**: Bursts and repeated quiet periods support irregular. The longest gap lasted 7 months, with 9 gaps of at least three months. The gap cause is difficult to establish from the messages.

**vearch_vearch**: Uneven activity, a long pause and a large 2024 burst support irregular. The longest gap lasted 7 months, with 2 gaps of at least three months. The gap cause is difficult to establish from the messages.

**hannorein_rebound**: Large bursts in 2015-2016 and 2023 interrupt quieter periods, supporting irregular. The longest gap lasted 3 months, with 1 gaps of at least three months. The gap cause is difficult to establish from the messages.

**soedinglab_mmseqs2**: Activity is lower after the 2016 peak, despite later fluctuations, supporting declining. The longest gap lasted 1 months, with 0 gaps of at least three months. The gap cause is difficult to establish from the messages.

**hpddm_hpddm**: Several peaks and quiet periods without a consistent long-term direction support irregular. The longest gap lasted 3 months, with 2 gaps of at least three months. The gap cause is difficult to establish from the messages.

**stephenslab_mr.mash.alpha**: Activity is much lower after the initial 2020 burst, supporting declining. The longest gap lasted 5 months, with 2 gaps of at least three months. The gap cause is difficult to establish from the messages.

**gem_oq-engine**: Monthly activity continues, but counts are lower after the 2018-2019 peak, supporting declining. There are no zero-commit months between the first and last retrieved commits; No recovery means no gap occurred. There is no inactivity gap to interpret.

**sixin-zh_kymatio_wph**: Most work occurred in 2019, followed by very sparse updates, supporting declining. The longest gap lasted 30 months, with 6 gaps of at least three months. The gap cause is difficult to establish from the messages.